In [42]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.metrics import r2_score

In [22]:
gold_dir = Path('../data/gold')
dim_produto = pd.read_parquet(gold_dir / 'dim_produto.parquet')[['sk_produto','tipo_credito']]
dim_consumidor = pd.read_parquet(gold_dir / 'dim_consumidor.parquet')[['sk_consumidor', 'faixa_etaria', 'faixa_score_credito', 'faixa_renda', 'flag_negativado']]
dim_tempo = pd.read_parquet(gold_dir / 'dim_tempo.parquet')

fact_credito = pd.read_parquet(gold_dir / 'gold_credito.parquet')[['sk_produto','sk_consumidor', 'sk_tempo', 'conversao_efetiva','taxa_conversao','possui_oferta']]
fact_credito = fact_credito.groupby(['sk_produto','sk_consumidor'], as_index=False).agg({'taxa_conversao':'mean','possui_oferta':'sum','conversao_efetiva':'sum'})

df = fact_credito.merge(dim_produto, on='sk_produto', how='left')
df = df.merge(dim_consumidor, on='sk_consumidor', how='left')
df

,sk_produto,sk_consumidor,taxa_conversao,possui_oferta,conversao_efetiva,tipo_credito,faixa_etaria,faixa_score_credito,faixa_renda,flag_negativado
0,1,1,0.050971,10869.0,554.0,conta_digital,0-20,301-400,0-1600,True
1,1,2,0.007333,28501.0,209.0,conta_digital,31-40,701-800,4001-5000,False
2,1,3,0.013875,7784.0,108.0,conta_digital,41-50,701-800,1601-2000,False
3,1,4,0.003628,1654.0,6.0,conta_digital,21-30,801-900,7001-8000,False
4,1,5,0.030186,52640.0,1589.0,conta_digital,31-40,401-500,4001-5000,True
...,...,...,...,...,...,...,...,...,...,...
5812,7,1175,0.500000,2.0,1.0,emprestimo_garantia_imovel,21-30,601-700,1601-2000,False
5813,7,1178,0.000000,3.0,0.0,emprestimo_garantia_imovel,41-50,801-900,8001-9000,False
5814,7,1182,0.000000,2.0,0.0,emprestimo_garantia_imovel,31-40,601-700,2001-3000,False
5815,7,1189,0.307692,13.0,4.0,emprestimo_garantia_imovel,31-40,501-600,4001-5000,False


In [23]:
# Sucessos / trials
df['trials'] = df['possui_oferta'].astype(float)
df['sucessos'] = df['conversao_efetiva'].astype(float)

In [31]:
# Preparar dados (o que você já tem no df_model)
X = df[['faixa_score_credito', 'tipo_credito', 'flag_negativado', 'faixa_etaria', 'faixa_renda']]
y = df['taxa_conversao']
weights = df['trials']

# Split (80/20)
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights, test_size=0.2, random_state=42, stratify=X['tipo_credito']
)

# Rodar o modelo só no treino
model_pred = smf.glm(
    formula='taxa_conversao ~ C(faixa_score_credito, Treatment("0-100")) + '
            'C(tipo_credito, Treatment("conta_digital")) + flag_negativado + '
            'C(faixa_etaria) + C(faixa_renda)',
    data=pd.concat([X_train, y_train], axis=1),   # juntar de novo
    family=sm.families.Binomial(),
    freq_weights=w_train
).fit(disp=False)

In [41]:
#checagem de p-valores
insignificant_vars = model_pred.pvalues[model_pred.pvalues >= 0.05]
print("Variáveis insignificativas (p >= 0.05):")
print(insignificant_vars)

if len(insignificant_vars) > 0:
    print("Existem variáveis insignificativas no modelo.")
else:
    print("Não há variáveis insignificativas no modelo.")
    print('Modelo estatisticamente significativo.')

Variáveis insignificativas (p >= 0.05):
Series([], dtype: float64)
Não há variáveis insignificativas no modelo.
Modelo estatisticamente significativo.


In [43]:
# Prever no teste
y_pred = model_pred.predict(X_test)

# Avaliar a performance glm - regressao 
print("R²:", r2_score(y_test, y_pred))

R²: 0.2770688660223304


#Construir modelo completo para prever taxa de conversão em todas as combinações possíveis de consumidor e produto

In [52]:
# Todas as combinações possíveis de consumidor e produto
df_all = dim_consumidor.merge(dim_produto, how='cross')
df_all

,sk_consumidor,faixa_etaria,faixa_score_credito,faixa_renda,flag_negativado,sk_produto,tipo_credito
0,1,0-20,301-400,0-1600,True,1,conta_digital
1,1,0-20,301-400,0-1600,True,2,emprestimo_pessoal_fgts
2,1,0-20,301-400,0-1600,True,3,emprestimo_pessoal
3,1,0-20,301-400,0-1600,True,4,emprestimo_garantia_veiculo
4,1,0-20,301-400,0-1600,True,5,cartao_credito
...,...,...,...,...,...,...,...
9200,1315,71-80,601-700,5001-6000,True,3,emprestimo_pessoal
9201,1315,71-80,601-700,5001-6000,True,4,emprestimo_garantia_veiculo
9202,1315,71-80,601-700,5001-6000,True,5,cartao_credito
9203,1315,71-80,601-700,5001-6000,True,6,emprestimo_consignado


In [79]:
# trazer as combinações consumidor-produto que ainda não têm oferta
df_sem_oferta = (
	df_all
	.merge(
		df[['sk_consumidor', 'sk_produto']],
		on=['sk_consumidor', 'sk_produto'],
		how='left',
		indicator=True
	)
	.query("_merge == 'left_only'")
	.drop(columns=['_merge'])
)

# Preparar dados (o que você já tem no df_model)
X = df[['faixa_score_credito', 'tipo_credito', 'flag_negativado', 'faixa_etaria', 'faixa_renda']]
y = df['taxa_conversao']
weights = df['trials']

# Rodar o modelo só no treino
model_pred = smf.glm(
    formula='taxa_conversao ~ C(faixa_score_credito, Treatment("0-100")) + '
            'C(tipo_credito, Treatment("conta_digital")) + flag_negativado + '
            'C(faixa_etaria) + C(faixa_renda)',
    data=pd.concat([X, y], axis=1),   # juntar de novo
    family=sm.families.Binomial(),
    freq_weights=weights
).fit(disp=False)

# Prever nos dados sem oferta
x_pred = df_sem_oferta[['faixa_score_credito', 'tipo_credito', 'flag_negativado', 'faixa_etaria', 'faixa_renda']]
y_pred = model_pred.predict(x_pred)


In [81]:
x_pred['taxa_conversao_prevista'] = y_pred

C:\Users\eduar\AppData\Local\Temp\ipykernel_17424\1596231883.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x_pred['taxa_conversao_prevista'] = y_pred


In [84]:
df_pred = (
            x_pred
            .merge(dim_consumidor, on=['faixa_etaria', 'faixa_score_credito', 'faixa_renda', 'flag_negativado'], how='left')
            .merge(dim_produto, on='tipo_credito', how='left')
            )

df_pred = df_pred[['sk_consumidor', 'sk_produto', 'taxa_conversao_prevista']]
df_pred.to_parquet(gold_dir / 'predicao_glm.parquet', index=False)
